In [2]:
# Scraping a sample job posting

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.myjobmag.co.ke/job/global-data-analyst-fixed-term-one-acre-fund?utm_campaign=google_jobs_apply&utm_source=google_jobs_apply&utm_medium=organic")
page_data = loader.load().pop().page_content
# print(page_data)

This project uses Groq’s ChatGroq API with the LLaMA 3.3 70B versatile model to build an AI assistant for tailoring resumes and generating cover letters. 

The JSON output parser enables reliable extraction of information and ensures the generated data can be programmatically processed.

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [5]:
# Initializing the LLM
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    temperature=0,
    groq_api_key=api_key,
    model_name="llama-3.3-70b-versatile"
)

# Adding the JSON Parser
parser = JsonOutputParser()

# 1. Job Matching

In [6]:
# Job Details Extraction Prompt

prompt_extract = PromptTemplate.from_template(
"""
You are an expert information extraction system.

### CONTEXT
The following text was scraped from a company's career page and contains
information about a job posting. The text may include formatting noise,
navigation elements, or repeated content.

### TASK
Extract the job information from the text.

Return a JSON object with the following fields:

- role: The job title
- experience: Required experience level (years or level such as junior, mid, senior)
- skills: List of key technical or domain skills required for the role
- description: A concise 2–5 sentence summary of the job responsibilities

### RULES
- Extract information only from the provided text.
- Do NOT invent information.
- If a field is missing, return null.
- Skills must be returned as a list of strings.
- Return ONLY a valid JSON object.
- Do not include explanations, markdown, backticks, or extra text

### JSON FORMAT
{{
  "role": "Job title",
  "experience": "Experience requirement",
  "skills": ["skill1", "skill2", "skill3"],
  "description": "Short description of the role"
}}

### SCRAPED TEXT
{page_data}
"""
)

# Building the extract chain
chain_extract = prompt_extract | llm | parser

In [7]:
# Running the chain extraction on the scraped jd
job_data = chain_extract.invoke({"page_data": page_data})
print(job_data)

{'role': 'Global Data Analyst (Fixed-Term)', 'experience': '3+ years', 'skills': ['SQL', 'Python', 'R', 'BI development', 'Data modeling', 'Data storytelling', 'Superset', 'Power BI', 'Dataiku'], 'description': 'The Global Data Analyst will build foundational, self-service BI data products, own analytical domains and partner with data engineering, and strengthen analytical standards across the organization. The role requires strong SQL skills, experience with BI tools, and the ability to communicate complex business logic into clear, usable data products.'}


# 2. Portfolio Projects Optimizer

In [8]:
import pandas as pd
import uuid
import chromadb

In [9]:
portfolio_df = pd.read_csv("portfolio.csv")
portfolio_df.head(2)

,project_name,description,tech_stack,link,source_page,all_text
0,JiBudget,Personal finance and budgeting web application...,"css, django, html, javascript, mysql, python",https://github.com/sharlynmuturi/JiBudget,https://sharlynmuturi.github.io/,JiBudget Personal finance and budgeting web ap...
1,Motor Insurance Database,Database-driven insurance management system su...,"css, django, html, sql",https://github.com/sharlynmuturi/motor-insuran...,https://sharlynmuturi.github.io/,Motor Insurance Database Database-driven insur...


In [10]:
# Initializing chromaDB and adding projects

# Create or connect to persistent vectorstore
client = chromadb.PersistentClient('vectorstore')

# Create or get collection
collection = client.get_or_create_collection(name="portfolio")

# Add portfolio projects only if collection is empty
if not collection.count():
    for _, row in portfolio_df.iterrows():
        collection.add(
            documents=[row["all_text"]],  # vectorizing the full text
            metadatas={
                "link": row.get("link", ""),
                "project_name": row.get("project_name", ""),
                "tech_stack": row.get("tech_stack", "")  # empty string if missing
            },
            ids=[str(uuid.uuid4())]
        )
        
print(f"Total projects in ChromaDB: {collection.count()}")

Total projects in ChromaDB: 18


In [11]:
job_description_text = job_data["description"]

# Querying chromaDB for relevant projects
results = collection.query(
    query_texts=[job_description_text],
    n_results=5  # top 5 most relevant projects
)

# Format top projects for the LLM
top_projects_text = ""
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    project_name = meta.get("project_name", "Unknown Project")
    tech_stack = meta.get("tech_stack", "")  # empty string if missing
    link = meta.get("link", "")

    top_projects_text += f"""
Project: {project_name}
Description: {doc}
Technologies: {tech_stack}
Link: {link}
"""

print(top_projects_text[:800])


Project: Product & Customer Sales Performance ETL
Description: Product & Customer Sales Performance ETL End-to-end customer and product sales analytics pipeline, including data ingestion, aggregation, feature engineering, performance analysis and visualization. 
Technologies: 
Link: https://github.com/sharlynmuturi/data-engineering-and-analytics-projects/tree/main/customer-product-sales-analytics-pipeline

Project: Sales Data Warehouse and Analytics
Description: Sales Data Warehouse and Analytics Star-schema data warehouse and analytical reporting using SQL and Tableau in evaluating sales performance trends. sql, tableau
Technologies: sql, tableau
Link: https://github.com/sharlynmuturi/data-engineering-and-analytics-projects/tree/main/data-warehouse-and-analytics

Project: HR Data Analysi


# 3. Tailor Resume

In [12]:
import pdfplumber

def read_resume(path="resume.pdf"):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

resume_text = read_resume()
print(resume_text[:1000])

Sharlyn Muturi
Portfolio:sharlynmuturi.github.io
(+254)748-736720 github.com/sharlynmuturi
DataScientist/Underwriter/WebDeveloper
sharlynnmuturi@gmail.com
Analyticalactuarialsciencegraduatewithhands-onexperienceininsuranceoperationsanddata-drivenproblemsolving.Strong
backgroundinstatisticalmodeling,machinelearningandtimeseriesanalysis,withpracticalexperienceinpolicyevaluation,risk
assessmentandclaimsanalysis.ProficientinPython,R,SQLandDjangoORMforstructureddatamanagement,analyticsand
applicationdevelopment.Motivatedtoapplyanalyticalskillstosolveindustrychallengesandsupportinformeddecision-making
indataandrisk-focusedenvironments.
SKILLS
ProgrammingandWeb Python,R,SQL,JavaScript,HTML/CSS,Django,Flask,Streamlit.
Data&Analytics Databricks,Tableau,PowerBI,SQLServer,MySQL,PostgreSQL,Spark,Excel,Access,A/Btesting.
MachineLearning RegressionandClassificationModels,TimeSeriesAnalysis,PredictiveModeling,NLP,NeuralNetworks,
LLMs(LangChain,HuggingFace,OpenAI).
InsuranceOperations Underwritingproc

In [15]:
# Resume Extraction Prompt

prompt_resume = PromptTemplate.from_template(
"""
You are an expert resume parser.

### TASK
Extract structured information from the resume text.

Return a JSON object with the following fields:

- name: candidate name
- education: list of education entries
- skills: list of technical skills
- experience: list of professional experiences
- projects: list of projects mentioned

### RULES
- Extract information only from the text.
- Do NOT invent information.
- If something is missing return null.
- Skills must be a list of strings.
- Return ONLY valid JSON.

### JSON FORMAT
{{
  "name": "Candidate name",
  "education": ["degree1", "degree2"],
  "skills": ["skill1", "skill2"],
  "experience": ["experience1", "experience2"],
  "projects": ["project1", "project2"]
}}

### RESUME TEXT
{resume_text}
"""
)

parser = JsonOutputParser()

chain_resume = prompt_resume | llm | parser

# Running the resume extraction
resume_data = chain_resume.invoke({"resume_text": resume_text})

print(resume_data)

{'name': 'Sharlyn Muturi', 'education': ['Bachelor of Science in Actuarial Science, GPA: 3.5, The University of Nairobi 2020—2024', 'Kenya Certificate of Secondary Education (K.C.S.E), Grade: A-, Chogoria Girls’ High School 2016—2019'], 'skills': ['Python', 'R', 'SQL', 'JavaScript', 'HTML/CSS', 'Django', 'Flask', 'Streamlit', 'Databricks', 'Tableau', 'PowerBI', 'SQL Server', 'MySQL', 'PostgreSQL', 'Spark', 'Excel', 'Access', 'A/B testing', 'Regression and Classification Models', 'Time Series Analysis', 'Predictive Modeling', 'NLP', 'Neural Networks', 'LLMs (LangChain, HuggingFace, OpenAI)', 'German'], 'experience': ['UNDERWRITING ASSISTANT, Incourage Insurance Agency, Nairobi, Kenya, Dec 2024—June 2025', 'INSURANCE OFFICER ATTACHEE, Kenya Electricity Generating Company PLC (KenGen), Nairobi, Kenya, May 2023—Jul 2023'], 'projects': ['JiBudget - Personal finance and budgeting web application', 'Motor Insurance System - System supporting policy issuance, underwriting workflows and data st

In [18]:
print("\nCandidate:", resume_data["name"])
print("\nSkills:", ", ".join(resume_data["skills"]))
print("\nEducation:", resume_data["education"])
print("\nProjects:", resume_data["projects"])


Candidate: Sharlyn Muturi

Skills: Python, R, SQL, JavaScript, HTML/CSS, Django, Flask, Streamlit, Databricks, Tableau, PowerBI, SQL Server, MySQL, PostgreSQL, Spark, Excel, Access, A/B testing, Regression and Classification Models, Time Series Analysis, Predictive Modeling, NLP, Neural Networks, LLMs (LangChain, HuggingFace, OpenAI), German

Education: ['Bachelor of Science in Actuarial Science, GPA: 3.5, The University of Nairobi 2020—2024', 'Kenya Certificate of Secondary Education (K.C.S.E), Grade: A-, Chogoria Girls’ High School 2016—2019']

Projects: ['JiBudget - Personal finance and budgeting web application', 'Motor Insurance System - System supporting policy issuance, underwriting workflows and data storage', 'Stock Price Forecasting - Stock Forecasting using Prophet and ARIMA models', 'Vehicle Insurance Fraud Detection - Rule-based scoring and unsupervised ML for anomaly detection', 'A/B testing & Experimentation - System using experimental design, hypothesis testing, power 

In [19]:
job_skills = set(job_data["skills"])
resume_skills = set(resume_data["skills"])

matching = job_skills.intersection(resume_skills)

print("Matching skills:", matching)

Matching skills: {'Python', 'R', 'SQL'}


In [13]:
# Resume tailoring prompt
prompt_tailor_resume = PromptTemplate.from_template(
"""
You are an expert career assistant and resume writer.

Your task is to tailor a candidate's resume for a specific job role.

### JOB DESCRIPTION
{job_data}

### CANDIDATE RESUME
{resume_data}

### CANDIDATE PROJECTS
{portfolio_projects}

### INSTRUCTIONS
Rewrite the candidate's resume so that it better aligns with the job.

Focus on:
- highlighting relevant skills
- emphasizing relevant projects
- matching the language used in the job description
- keeping all information truthful

### OUTPUT FORMAT

Return a professional resume with the following sections:

NAME

PROFESSIONAL SUMMARY

SKILLS

EXPERIENCE

PROJECTS

EDUCATION

The resume should be concise, professional, and optimized for ATS systems.

Return only the resume text.
"""
)

chain_tailor_resume = prompt_tailor_resume | llm

In [16]:
tailored_resume = chain_tailor_resume.invoke({
    "job_data": job_data,
    "resume_data": resume_text,
    "portfolio_projects": top_projects_text
})

print("===== TAILORED RESUME =====\n")
print(tailored_resume.content)

===== TAILORED RESUME =====

Sharlyn Muturi

PROFESSIONAL SUMMARY
Results-driven data analyst with 3+ years of experience in data-driven problem-solving, leveraging strong SQL skills, and proficiency in BI tools to drive business growth. Proven ability to communicate complex business logic into clear, usable data products, with a strong background in statistical modeling, machine learning, and data storytelling.

SKILLS
- Programming: Python, R, SQL, JavaScript, HTML/CSS
- Data & Analytics: SQL, Tableau, Power BI, Databricks, Excel
- Data Modeling: Data warehousing, star-schema design
- BI Development: BI tools, data visualization, reporting
- Data Storytelling: Communicating complex data insights to stakeholders

EXPERIENCE
UNDERWRITING ASSISTANT
Incourage Insurance Agency, Nairobi, Kenya
Dec 2024 - Jun 2025
- Analyzed risk data, including transactional and premium reconciliation, utilizing SQL for data management
- Developed and maintained accurate insurance quotes and certificates, 

In [17]:
with open("tailored_resume.txt", "w", encoding="utf-8") as f:
    f.write(tailored_resume.content)

# 4. Cover Letter Generation

In [18]:
prompt_cover_letter = PromptTemplate.from_template("""
You are an expert career assistant.

### CONTEXT
You have the following information:

- Job posting details: {job_data}
- Candidate resume content: {resume_data}
- Relevant portfolio projects: {portfolio_projects}

### TASK
Write a professional and persuasive **cover letter** tailored to this specific job. 
The cover letter should:

1. Address the hiring manager (use "Dear Hiring Manager" if name is unknown)
2. Highlight the candidate's most relevant skills and experience from the resume
3. Reference the most relevant portfolio projects
4. Match the tone of the job posting (formal, professional)
5. Be 3–5 paragraphs long
6. End with a polite call-to-action

### RULES
- Only use information provided in the context; do not invent details
- Keep it concise and impactful
- Output plain text (no JSON)
""")

chain_cover_letter = prompt_cover_letter | llm

res_cover_letter = chain_cover_letter.invoke({
    "job_data": job_data,
    "resume_data": resume_text,
    "portfolio_projects": top_projects_text
})

# output
cover_letter_text = res_cover_letter["content"] if isinstance(res_cover_letter, dict) else res_cover_letter.content

print("===== AI GENERATED COVER LETTER =====\n")
print(cover_letter_text)

===== AI GENERATED COVER LETTER =====

Dear Hiring Manager,

I am excited to apply for the Global Data Analyst (Fixed-Term) position, where I can utilize my analytical skills to drive informed decision-making in a data-focused environment. With over 3 years of experience in data-driven problem-solving and a strong background in statistical modeling, machine learning, and time series analysis, I am confident in my ability to make a significant impact in this role.

As a detail-oriented and motivated individual, I possess a unique combination of technical skills, including proficiency in Python, R, SQL, and experience with BI tools such as Tableau and Power BI. My experience in data modeling, data storytelling, and BI development is evident in my portfolio projects, particularly in the Product & Customer Sales Performance ETL and Sales Data Warehouse and Analytics projects. These projects demonstrate my ability to design and implement end-to-end analytics pipelines, perform data aggregat

In [19]:
with open("cover_letter.txt", "w", encoding="utf-8") as f:
    f.write(cover_letter_text)